# Recolección de telemetría de tráfico

Workflow opcional y reutilizable de adquisición de datos de VAAET ML 4.0.0. Procesa un clip una sola vez y produce un video anotado y telemetría cruda por minuto. Puede acumular el CSV canónico y, si se habilita explícitamente, persistir los registros en PostgreSQL.

Los nombres con formato `bridge_YYYY-MM-DD_HH-MM-SS_to_HH-MM-SS.mp4` conservan la hora real de captura. Para nombres libres se utiliza la hora de procesamiento, con menor trazabilidad temporal.

In [ ]:
# Environment setup — run once per Colab runtime
import importlib.metadata
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/zgfnicolas/vaaet.git"
REPO_DIR = Path("/content/vaaet")

if IN_COLAB:
    if (REPO_DIR / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
    REPO_ROOT = REPO_DIR.resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next(
        (path for path in candidates if (path / "pyproject.toml").is_file() and (path / "src/vaaet").is_dir()),
        None,
    )
    if REPO_ROOT is None:
        raise RuntimeError("VAAET repository root not found")

os.chdir(REPO_ROOT)
install_command = [sys.executable, "-m", "pip", "install", "-q"]
if IN_COLAB:
    install_command.append(f"{REPO_ROOT}[vision,database]")
else:
    install_command.extend(["-e", f"{REPO_ROOT}[vision,database]"])
subprocess.check_call(install_command)

for module_name in tuple(sys.modules):
    if module_name == "vaaet" or module_name.startswith("vaaet."):
        sys.modules.pop(module_name, None)
importlib.invalidate_caches()

import vaaet

def validate_vaaet_origin(package: object, repo_root: Path, in_colab: bool) -> Path:
    package_file = getattr(package, "__file__", None)
    if not package_file:
        package_path = list(getattr(package, "__path__", ()))
        raise ImportError(
            "The 'vaaet' import resolved to a namespace package instead of the installed package. "
            f"Resolved locations: {package_path}. Re-run this setup cell."
        )
    origin = Path(package_file).resolve()
    expected_editable_root = (repo_root / "src/vaaet").resolve()
    if in_colab and repo_root.resolve() in origin.parents:
        raise ImportError(f"Colab must load the installed wheel, not repository path: {origin}")
    if not in_colab and origin.parent != expected_editable_root:
        raise ImportError(f"Local editable install has unexpected origin: {origin}")
    return origin

VAAET_PACKAGE_FILE = validate_vaaet_origin(vaaet, REPO_ROOT, IN_COLAB)
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True,
    check=False,
)
pip_check_output = "\n".join(
    part.strip() for part in (pip_check.stdout, pip_check.stderr) if part.strip()
)
if pip_check.returncode == 0:
    print("✅ pip check: no broken requirements found")
else:
    print("⚠️ pip check detected conflicts in the managed notebook runtime:")
    print(pip_check_output or "No diagnostic output was returned")
    print("ℹ️ Continuing because workflow imports are validated explicitly below.")

def package_version(name: str) -> str:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return "not required"

print({name: package_version(name) for name in ("numpy", "tensorflow", "opencv-python-headless", "ultralytics-opencv-headless")})

import cv2
import numpy as np
import pandas as pd
import psycopg2
import sqlalchemy
import torch
import ultralytics

from vaaet.data.database import get_optional_db_config, hydrate_db_environment_from_colab
from vaaet.data.datasets import merge_raw_telemetry_csv
from vaaet.data.persistence import persist_raw_telemetry
from vaaet.logging import configure_logging
from vaaet.vision.analysis import analyze_video

hydrate_db_environment_from_colab()
configure_logging()
print(f"Python {sys.version.split()[0]} | NumPy {np.__version__} | OpenCV {cv2.__version__} | Ultralytics {ultralytics.__version__} | GPU {torch.cuda.is_available()}")
print(f"Package: {VAAET_PACKAGE_FILE}")
print(f"✅ data collection workflow ready | root={REPO_ROOT} | commit={subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()}")


## 1. Seleccionar el clip

En Colab se abre el selector de archivos. En ejecución local, coloque un ejemplo no sensible en `data/sample/` o asigne una ruta absoluta a `VIDEO_PATH`.

In [ ]:
VIDEO_PATH: Path | None = None
if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = Path(next(iter(uploaded))).resolve()
else:
    candidate = REPO_ROOT / "data/sample/sample.mp4"
    VIDEO_PATH = candidate if candidate.is_file() else None

print(f"Clip: {VIDEO_PATH or 'not selected'}")

## 2. Analizar y acumular resultados

YOLO descarga sus pesos oficiales durante la primera inferencia; los pesos no se guardan en Git. La función compartida produce el HUD en modo **Telemetry Collection** y evita duplicar el motor visual dentro del notebook.

In [ ]:
if VIDEO_PATH is None or not VIDEO_PATH.is_file():
    raise FileNotFoundError("Select or upload a valid MP4 clip before continuing")

OUTPUT_VIDEO = Path("/content") / f"{VIDEO_PATH.stem}_vaaet_annotated.mp4" if IN_COLAB else VIDEO_PATH.with_name(f"{VIDEO_PATH.stem}_vaaet_annotated.mp4")
result = analyze_video(VIDEO_PATH, OUTPUT_VIDEO)

RAW_CSV = REPO_ROOT / "data/raw/traffic_data_raw.csv"
df_raw = merge_raw_telemetry_csv(result.telemetry, RAW_CSV)
display(result.telemetry)
print(f"✅ Annotated video: {result.video_path}")
print(f"✅ Canonical CSV: {RAW_CSV} ({len(df_raw)} unique rows)")

if IN_COLAB:
    from google.colab import files

    files.download(str(result.video_path))
    files.download(str(RAW_CSV))

## 3. Persistencia PostgreSQL opcional

La persistencia está deshabilitada por defecto. Configure `DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER` y `DB_PASSWORD` en **Colab Secrets** (o como variables de entorno) y cambie `PERSIST_TO_DATABASE` a `True`. La clave `(clip_id, record_time)` hace la operación idempotente.

In [ ]:
PERSIST_TO_DATABASE = False

if PERSIST_TO_DATABASE:
    config = get_optional_db_config(interactive=False)
    if config is None:
        raise RuntimeError("Database credentials are not configured")
    inserted = persist_raw_telemetry(result.telemetry, config=config)
    print(f"✅ Inserted {inserted} new rows into traffic_data")
else:
    print("ℹ️ Database persistence disabled")